# 02. Sentiment Modeling: Detecting Information Asymmetry

**Context**: In the Kenyan e-commerce landscape, numerical star ratings often mask qualitative dissatisfaction. This notebook builds an NLP pipeline to identify "Information Asymmetry"—cases where products maintain high ratings despite critical feedback hidden in local dialects (Sheng/Swahili).

## 2.1 Load Preprocessed Data

We pull the synthesized NLP dataset created in Phase 1. This dataset already includes slang-mapped tokens and joined product metadata.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_preprocessing import DataLoader, TextPreprocessor, DataCleaner
from src.sentiment_analysis import SentimentModel
from sklearn.model_selection import train_test_split

# Initialize tools
loader = DataLoader(data_dir='../data/raw/')
text_proc = TextPreprocessor()

# Load and build the NLP frame
_, prods, revs = loader.load_all()
nlp_df = text_proc.build_nlp_frame(revs, prods)

print(f"NLP Frame Ready: {nlp_df.shape[0]} samples")

NLP Frame Ready: 92 samples


## 2.2 Vectorization & Training

We use TF-IDF (Term Frequency-Inverse Document Frequency) to convert our tokens into a numerical matrix. By using ngram_range=(1, 2), we capture both individual words (e.g., "feki") and phrases (e.g., "not good").

In [2]:
# Initialize our model class from src/sentiment_analysis.py
sm = SentimentModel(max_features=2500)

# 1. Transform text to features
X_tfidf = sm.prepare_features(nlp_df["tokens"])
y = nlp_df["sentiment_target"]

# 2. Train-Test Split (Stratified to maintain sentiment balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Fit the Classifier
sm.train(X_train, y_train)

# 4. Evaluation (This will print the report)
sm.evaluate(X_test, y_test)

--- Sentiment Engine: Detailed Report ---
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         1
         1.0       0.95      1.00      0.97        18

    accuracy                           0.95        19
   macro avg       0.47      0.50      0.49        19
weighted avg       0.90      0.95      0.92        19



array([[ 0,  1],
       [ 0, 18]])

## 2.3 Model Persistence

To use this model in our final integration (Phase 4), we must persist the trained objects to the models/ directory.

In [3]:
# This creates 'sentiment_rf.joblib' and 'tfidf_vec.joblib' in /models
sm.save()

☑ Success: Sentiment components saved to ../models/sentiment_rf.joblib and ../models/tfidf_vec.joblib
